In [1]:
# Parameters
input_file = "/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/CASTEP/faux_D-alanine_295K_278464_c-inv_unopt_pbesol_SEDC_magres.magres"


*This code uses papermill on terminal to change input_file. The input_file is the file path to the magres file*

*Shiva Agarwal*

*Apr 23 2025*

In [2]:
import os
import numpy as np
from tabulate import tabulate
from magres.atoms import MagresAtoms

atoms = MagresAtoms.load_magres(input_file)


In [3]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [4]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [5]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g
    

In [6]:
nucleus = 'N'      # nucleus for which parameters are wanted
atom_label = 0      # site for which parameters wanted
Q = 0.0204 #electric quadrupole moment for 14N in barn

In [7]:
for atom in atoms.species('N'):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()

14N1 sigma:
 [[188.76556004   1.77861214   4.1357836 ]
 [ -1.28613208 176.69582897   2.98032618]
 [  4.76985737   3.9250928  193.51814629]]

14N2 sigma:
 [[188.76556004  -1.77861214  -4.1357836 ]
 [  1.28613208 176.69582897   2.98032618]
 [ -4.76985737   3.9250928  193.51814629]]

14N3 sigma:
 [[188.76556004   1.77861214  -4.1357836 ]
 [ -1.28613208 176.69582897  -2.98032618]
 [ -4.76985737  -3.9250928  193.51814629]]

14N4 sigma:
 [[188.76556004  -1.77861214   4.1357836 ]
 [  1.28613208 176.69582897  -2.98032618]
 [  4.76985737  -3.9250928  193.51814629]]



In [8]:
for atom in atoms.species('N'):
    print (atom, "sigma:\n",atom.efg.Cq)
    print()

14N1 sigma:
 1.2898163462001966

14N2 sigma:
 1.2898163462002197

14N3 sigma:
 1.2898163462002101

14N4 sigma:
 1.289816346200208



In [9]:
# using values from latest magres file
Cs = np.zeros((3, 3))                                # CS symmetric (l = 0 + 2) Tensor from updated_magres
CS_anti = np.zeros((3,3))                           # CS antisymmetric ( l = 1)
CS_iso = np.zeros((3,3))                            # CS isotropic  (l = 0)
CS_total = np.zeros((3,3))                            # CS total shielding tensor ( l = 0 + 1 + 2) Tensor from magres

                                         
CS_total[:,:] = atoms.species(nucleus).ms.sigma[atom_label]

iso = np.mean([CS_total[0,0], CS_total[1,1], CS_total[2,2]]) # isotropic chemical shielding (l = 0)

CS_iso[0,0] = CS_iso[1,1] = CS_iso[2,2] = iso

Cs[0,0] = CS_total[0,0]; Cs[0,1] = (CS_total[0,1] + CS_total[1,0] )/2; Cs[0,2] = (CS_total[0,2] + CS_total[2,0])/2;
Cs[1,0] = Cs[0,1];      Cs[1,1] = CS_total[1,1];                      Cs[1,2] = (CS_total[1,2] + CS_total[2,1])/2;
Cs[2,0] = Cs[0,2];      Cs[2,1] = Cs[1,2];                           Cs[2,2] = CS_total[2,2];

CS_anti[0,1] = (CS_total[0,1] - CS_total[1,0])/2; CS_anti[0,2] = (CS_total[0,2] - CS_total[2,0])/2; 
CS_anti[1,0] = -CS_anti[0,1]; CS_anti[1,2] = (CS_total[1,2] - CS_total[2,1])/2;
CS_anti[2,0] = -CS_anti[0,2];      CS_anti[2,1] = -CS_anti[1,2]; 


efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)

efg[:,:] = atoms.species(nucleus)[atom_label].efg.V


# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 
# Q = 0.04059 barn https://www-nds.iaea.org/publications/indc/indc-nds-0650.pdf

V = efg*Q*234.9647

print('\nQ tensor:\n', np.round(V,3))
print('\nCS Tensor:\n',np.round(CS_total, 3))
print('\nCS isotropic Tensor:\n',np.round(CS_iso, 3))
print('\nCS symmetric Tensor:\n',np.round(Cs,3))
print('\nCS antisymmetric Tensor:\n',np.round(CS_anti,3))



Q tensor:
 [[ 0.433 -0.236  0.975]
 [-0.236 -0.421 -0.283]
 [ 0.975 -0.283 -0.012]]

CS Tensor:
 [[188.766   1.779   4.136]
 [ -1.286 176.696   2.98 ]
 [  4.77    3.925 193.518]]

CS isotropic Tensor:
 [[186.327   0.      0.   ]
 [  0.    186.327   0.   ]
 [  0.      0.    186.327]]

CS symmetric Tensor:
 [[188.766   0.246   4.453]
 [  0.246 176.696   3.453]
 [  4.453   3.453 193.518]]

CS antisymmetric Tensor:
 [[ 0.     1.532 -0.317]
 [-1.532  0.    -0.472]
 [ 0.317  0.472  0.   ]]


In [10]:
print("For EFG tensor")
sorted_eigenvalues_efg, dc_efg, quad_avg, eigenvalues_efg, eigenvectors_efg = sort_eigenvalues(V)
print('==================================\n')
print("For CS tensor")
sorted_eigenvalues_cs, dc_cs, cs_avg, eigenvalues_cs, eigenvectors_cs = sort_eigenvalues(Cs)

For EFG tensor
 Unsorted Eigenvalues:
 [ 1.2872918  -0.80649726 -0.48079454] 

 Unsorted Eigenvectors:
 [[-0.76020903  0.5756278   0.30122228]
 [ 0.20722978 -0.2245817   0.95216536]
 [-0.61574187 -0.78626694 -0.05144182]] 

Sorted Eigenvalues: 
 [-0.48079454 -0.80649726  1.2872918 ] 

Sorted Eigenvectors: 
 [[ 0.30122228  0.5756278  -0.76020903]
 [ 0.95216536 -0.2245817   0.20722978]
 [-0.05144182 -0.78626694 -0.61574187]] 


For CS tensor
 Unsorted Eigenvalues:
 [196.67188726 186.32573459 175.98191344] 

 Unsorted Eigenvectors:
 [[ 0.48848136  0.87097122 -0.05286858]
 [ 0.15445756 -0.14594141 -0.97716118]
 [ 0.85879498 -0.46915907  0.20581775]] 

Sorted Eigenvalues: 
 [186.32573459 175.98191344 196.67188726] 

Sorted Eigenvectors: 
 [[ 0.87097122 -0.05286858  0.48848136]
 [-0.14594141 -0.97716118  0.15445756]
 [-0.46915907  0.20581775  0.85879498]] 



In [11]:

#Calculate Quadrupolar Tensor in PAS

Vyy = sorted_eigenvalues_efg[0]
Vxx = sorted_eigenvalues_efg[1]
Vzz = sorted_eigenvalues_efg[2]

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================')

#Calculate CSA Tensor in PAS

Csyy = sorted_eigenvalues_cs[0] 
Csxx = sorted_eigenvalues_cs[1]  
Cszz = sorted_eigenvalues_cs[2]



print('CSA Tensor Components δyy, δxx, δzz: \n', Csyy, Csxx, Cszz)

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 -0.48079454001265276 -0.8064972627012275 1.2872918027139197
CSA Tensor Components δyy, δxx, δzz: 
 186.32573459302034 175.98191344420152 196.67188726463039


In [12]:
iso_cs = (Csxx + Csyy + Cszz)/3

csa = Cszz - iso_cs
etas = (Csyy - Csxx)/csa

#for Quadrupolar
CQ_fit = Vzz

etaq = (Vyy - Vxx)/Vzz

table = [['CQ (MHz)', CQ_fit], ['etaq', etaq ], ['iso_cs (ppm)',iso_cs ],['csa (ppm)', csa],  ['etas', etas]  ]

table_string = tabulate(table, headers=['Quantity', 'Value'], tablefmt='grid')
print(f'Parameters for {atoms.species(nucleus)[atom_label]}: \n', table_string)

Parameters for 14N1: 
 +--------------+------------+
| Quantity     |      Value |
+==============+============+
| CQ (MHz)     |   1.28729  |
+--------------+------------+
| etaq         |   0.253014 |
+--------------+------------+
| iso_cs (ppm) | 186.327    |
+--------------+------------+
| csa (ppm)    |  10.3454   |
+--------------+------------+
| etas         |   0.99985  |
+--------------+------------+


In [13]:

# Derive output name from input file
base_name = os.path.splitext(os.path.basename(input_file))[0]
output_txt = f"/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/NMR_analysis/output_txt/{nucleus}_all_results.txt"

# Save to .txt file
with open(output_txt, 'a') as f:
    f.write(f"\n\n===== Results for: {base_name} =====\n\n")
    f.write(table_string)
    f.write("\n")

print(f"Saved table to {output_txt}")

Saved table to /home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/NMR_analysis/output_txt/N_all_results.txt


In [14]:
# Calculation for efg tensor
print('Direction cosine efg:\n')
print(dc_efg, '\n')
a_efg, b_efg, g_efg = get_euler_angles(dc_efg)

print("Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:")
print(a_efg, b_efg, g_efg, '\n')

print('=========================')
print('Direction cosine csa: \n')
print(dc_cs, '\n')
a_cs, b_cs, g_cs = get_euler_angles(dc_cs)

print("Calculated Euler angles (degrees) CSA PAS --> Crystal:")
print(a_cs, b_cs, g_cs, '\n')

Direction cosine efg:

[[ 0.5756278   0.30122228 -0.76020903]
 [-0.2245817   0.95216536  0.20722978]
 [-0.78626694 -0.05144182 -0.61574187]] 

Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:
3.743263679856898 128.00584532489248 15.248106310362553 

Direction cosine csa: 

[[-0.05286858  0.87097122  0.48848136]
 [-0.97716118 -0.14594141  0.15445756]
 [ 0.20581775 -0.46915907  0.85879498]] 

Calculated Euler angles (degrees) CSA PAS --> Crystal:
-66.31317782128392 30.8184486072197 -17.54692808972994 



In [15]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_efg), (dc_cs))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: 54.38394248524221 chi: 150.24272054684008 xi: 30.249479582597036 

